### Load packages

In [13]:
import pandas as pd
import os
import numpy as np

import matplotlib.pyplot as plt


### Set file information

In [14]:
# USGS NWIS file info

usgs_directory = r"..\data\raw_files"
usgs_file = "USGS_NWIS_data_2026_02_09"

usgs_path = os.path.join(usgs_directory, usgs_file + ".csv")
print(usgs_path)

..\data\raw_files\USGS_NWIS_data_2026_02_09.csv


In [15]:
# USGS site location file info

site_loc_directory = r"..\data\data_dictionaries"
site_loc_file = "USGS_NWIS_site_locations"

site_loc_path = os.path.join(site_loc_directory, site_loc_file + ".csv")
print(site_loc_path)

..\data\data_dictionaries\USGS_NWIS_site_locations.csv


In [16]:
# Output file info

output_directory = r"..\data\unified_csvs"
output_file = "usgs_prepped"

output_path = os.path.join(output_directory, output_file + ".csv")
print(output_path)

..\data\unified_csvs\usgs_prepped.csv


### Read in data

In [17]:
# Read in USGS site location info

site_raw_df = pd.read_csv(site_loc_path)
site_raw_df.head()

,Unnamed: 0,site_no,station_nm,dec_lat_va,dec_long_va,site_tp_cd
0,1,1129400,"BLACK BROOK AT AVERILL, VT",45.003935,-71.692314,ST
1,2,1129420,"CAPON BROOK AT VT 102, NEAR CANAAN, VT",44.936944,-71.526111,ST
2,3,1129700,"PAUL STREAM TRIBUTARY NEAR BRUNSWICK SPRINGS, VT",44.685051,-71.621199,ST
3,4,1133000,"EAST BRANCH PASSUMPSIC RIVER NEAR EAST HAVEN, VT",44.633942,-71.897594,ST
4,5,1133100,"DISH MILL BROOK AT EAST BURKE, VT",44.589776,-71.922872,ST


In [18]:
# Clean up site location columns

site_cleancols_df = site_raw_df.copy()

col_map = {
    "site_no": "site",
    "dec_lat_va": "latitude",
    "dec_long_va": "longitude"
}

site_cleancols_df.rename(columns=col_map, inplace=True)

# Force site to integer
site_cleancols_df["site"] = (
    pd.to_numeric(site_cleancols_df["site"], errors="coerce")
        .astype("Int64")
)

site_cleancols_df.head()

,Unnamed: 0,site,station_nm,latitude,longitude,site_tp_cd
0,1,1129400,"BLACK BROOK AT AVERILL, VT",45.003935,-71.692314,ST
1,2,1129420,"CAPON BROOK AT VT 102, NEAR CANAAN, VT",44.936944,-71.526111,ST
2,3,1129700,"PAUL STREAM TRIBUTARY NEAR BRUNSWICK SPRINGS, VT",44.685051,-71.621199,ST
3,4,1133000,"EAST BRANCH PASSUMPSIC RIVER NEAR EAST HAVEN, VT",44.633942,-71.897594,ST
4,5,1133100,"DISH MILL BROOK AT EAST BURKE, VT",44.589776,-71.922872,ST


In [19]:
# Read in USGS csv data file

usgs_raw_df = pd.read_csv(usgs_path)

print(usgs_raw_df.columns)
usgs_raw_df.head()

Index(['Unnamed: 0', 'agency_cd', 'site_no', 'datetime', '219530_00010_00001',
       '219530_00010_00001_cd', '219535_00010_00002', '219535_00010_00002_cd',
       '325162_00010_00003', '325162_00010_00003_cd', '65129_00095_00001',
       '65129_00095_00001_cd', '65130_00095_00002', '65130_00095_00002_cd',
       '65131_00095_00003', '65131_00095_00003_cd'],
      dtype='object')


,Unnamed: 0,agency_cd,site_no,datetime,219530_00010_00001,219530_00010_00001_cd,219535_00010_00002,219535_00010_00002_cd,325162_00010_00003,325162_00010_00003_cd,65129_00095_00001,65129_00095_00001_cd,65130_00095_00002,65130_00095_00002_cd,65131_00095_00003,65131_00095_00003_cd
0,0,5s,15s,20d,14n,10s,14n,10s,14n,10s,14n,10s,14n,10s,14n,10s
1,1,USGS,04294500,2014-09-30,18.7,A,17.2,A,17.9,A,NaN,NaN,NaN,NaN,NaN,NaN
2,2,USGS,04294500,2014-10-01,17.2,A,15.8,A,16.5,A,179,A,170,A,173,A
3,3,USGS,04294500,2014-10-02,17.0,A,15.8,A,16.4,A,177,A,170,A,174,A
4,4,USGS,04294500,2014-10-03,17.3,A,16.5,A,16.9,A,179,A,173,A,175,A


### Format data & columns

In [20]:
# Drop the first row of data (it holds the units for each column)

print(len(usgs_raw_df))
usgs_nounits_df = usgs_raw_df.copy()

usgs_nounits_df = usgs_nounits_df[usgs_nounits_df['Unnamed: 0'] != 0]
print(len(usgs_nounits_df))

3006
3005


In [21]:
# Easy column cleanup

usgs_cleancols_df = usgs_nounits_df.copy()

# Drop the "Unnamed: 0" column (it's an artifact of something from the pull)
usgs_cleancols_df = usgs_cleancols_df[usgs_cleancols_df.columns.drop(["Unnamed: 0"])]

# Drop the "_cd" code columns
usgs_cleancols_df = usgs_cleancols_df[usgs_cleancols_df.columns.drop(list(usgs_cleancols_df.filter(regex='_cd$')))]

# Rename the data value columns by removing first 5 digits + underscore
def rename_col(col):
    if '_' in col and col.split('_')[0].isdigit():
        parts = col.split('_')
        # Remove first 5 digits part
        return '_'.join(parts[1:])  
    return col
usgs_cleancols_df = usgs_cleancols_df.rename(columns=rename_col)

usgs_cleancols_df.columns

Index(['site_no', 'datetime', '00010_00001', '00010_00002', '00010_00003',
       '00095_00001', '00095_00002', '00095_00003'],
      dtype='object')

In [22]:
# Turn all columns into lower case and then give some of them nicer names

usgs_cleancols_df.columns = usgs_cleancols_df.columns.str.lower()

col_map = {
    "site_no": "site",
    "datetime": "usgs_report_date",
}

usgs_cleancols_df.columns = (
    usgs_cleancols_df.columns
        .str.replace("^00010_", "usgs_water_temp_", regex=True)
        .str.replace("^00095_", "usgs_conductivity_", regex=True)
        .str.replace("00001", "max", regex=False)
        .str.replace("00002", "min", regex=False)
        .str.replace("00003", "mean", regex=False)
)

usgs_cleancols_df.rename(columns=col_map, inplace=True)

usgs_cleancols_df.head()

,site,usgs_report_date,usgs_water_temp_max,usgs_water_temp_min,usgs_water_temp_mean,usgs_conductivity_max,usgs_conductivity_min,usgs_conductivity_mean
1,04294500,2014-09-30,18.7,17.2,17.9,NaN,NaN,NaN
2,04294500,2014-10-01,17.2,15.8,16.5,179,170,173
3,04294500,2014-10-02,17.0,15.8,16.4,177,170,174
4,04294500,2014-10-03,17.3,16.5,16.9,179,173,175
5,04294500,2014-10-04,17.1,16.4,16.8,191,161,176


In [23]:
# Make sure it's only one site

usgs_cleancols_df["site"].value_counts()

site
04294500    3005
Name: count, dtype: int64

In [24]:
# Find any duplicates on date

usgs_cleancols_df["usgs_report_date"].duplicated().any()

np.False_

In [25]:
# Force dates to MM/DD/YYYY format

usgs_cleancols_df["usgs_report_date"] = (
    pd.to_datetime(usgs_cleancols_df["usgs_report_date"], errors="coerce")
      .dt.strftime("%m/%d/%Y")
)

usgs_cleancols_df["usgs_report_date"] = pd.to_datetime(
    usgs_cleancols_df["usgs_report_date"],
    errors="coerce"
)

print(usgs_cleancols_df["usgs_report_date"].min())
print(usgs_cleancols_df["usgs_report_date"].max())

2014-09-30 00:00:00
2022-12-31 00:00:00


In [26]:
# Force integers columns to integers

usgs_cleancols_df["site"] = (
    pd.to_numeric(usgs_cleancols_df["site"], errors="coerce")
        .astype("Int64")
)

### Merge latitude and longitude into the data

In [27]:
# QC site data
print(site_cleancols_df.columns)
print(usgs_cleancols_df.columns)
site_cleancols_df["site"].duplicated().any()

Index(['Unnamed: 0', 'site', 'station_nm', 'latitude', 'longitude',
       'site_tp_cd'],
      dtype='object')
Index(['site', 'usgs_report_date', 'usgs_water_temp_max',
       'usgs_water_temp_min', 'usgs_water_temp_mean', 'usgs_conductivity_max',
       'usgs_conductivity_min', 'usgs_conductivity_mean'],
      dtype='object')


np.True_

In [30]:
# Check out duplicate sites

dupe_sites_df = site_cleancols_df[
    site_cleancols_df["site"].duplicated(keep=False)
].sort_values("site")

dupe_sites_df

,Unnamed: 0,site,station_nm,latitude,longitude,site_tp_cd
4210,4237,443610073121502,"VT-CNW 131HALL, ROBERT NEW",44.602863,-73.203850,GW
4211,4238,443610073121502,VT-CNW 121,44.602863,-73.203850,GW
4213,4240,443610073122201,"VT-CNW 133LANPHERE, ROY",44.602863,-73.205794,GW
4214,4241,443610073122201,VT-CNW 117,44.602863,-73.205794,GW
4396,4423,443752073105301,"VT-ROWLEY,JAMES WR157 MJW 2",44.631196,-73.181071,GW
4397,4424,443752073105301,VT-MJW 29,44.631196,-73.181071,GW


In [31]:
# Merge latitude and longitude into USGS data

site_renamed_df = site_cleancols_df.copy()

site_renamed_df.columns = site_renamed_df.columns.str.lower()

col_map = {
    "latitude": "usgs_latitude",
    "longitude": "usgs_longitude",
}

site_renamed_df.rename(columns=col_map, inplace=True)

usgs_latlon_df = pd.merge(
    usgs_cleancols_df,
    site_renamed_df[["site", "usgs_latitude", "usgs_longitude"]],
    how="left",
    on="site"
)

usgs_latlon_df.head()

,site,usgs_report_date,usgs_water_temp_max,usgs_water_temp_min,usgs_water_temp_mean,usgs_conductivity_max,usgs_conductivity_min,usgs_conductivity_mean,usgs_latitude,usgs_longitude
0,4294500,2014-09-30,18.7,17.2,17.9,NaN,NaN,NaN,44.47616,-73.221517
1,4294500,2014-10-01,17.2,15.8,16.5,179,170,173,44.47616,-73.221517
2,4294500,2014-10-02,17.0,15.8,16.4,177,170,174,44.47616,-73.221517
3,4294500,2014-10-03,17.3,16.5,16.9,179,173,175,44.47616,-73.221517
4,4294500,2014-10-04,17.1,16.4,16.8,191,161,176,44.47616,-73.221517


### Building trailing columns for features

In [ ]:
# Compute trailing days for features

usgs_trailing_df = usgs_latlon_df.copy()

# Grouping by site & report_date
usgs_trailing_df["report_date"] = pd.to_datetime(usgs_trailing_df["report_date"])
usgs_trailing_df = usgs_trailing_df.sort_values(["report_date"])

trailing_days = 14

trailing_cont_features = ["feature_water_temp_max", "feature_water_temp_min", "feature_water_temp_mean", 
                          "feature_conductivity_max", "feature_conductivity_min", "feature_conductivity_mean"]

for col in trailing_cont_features:
    new_col = f"{col}_trailing"
    usgs_trailing_df[col] = pd.to_numeric(usgs_trailing_df[col], errors="coerce")

    usgs_trailing_df[new_col] = (
        usgs_trailing_df
        .groupby("site")[col]
        .rolling(trailing_days, min_periods=1)
        .mean()
        .shift(1)
        .reset_index(level=0, drop=True) 
    )

usgs_trailing_df.head(10)

,site,report_date,feature_water_temp_max,feature_water_temp_min,feature_water_temp_mean,feature_conductivity_max,feature_conductivity_min,feature_conductivity_mean,latitude,longitude,feature_water_temp_max_trailing,feature_water_temp_min_trailing,feature_water_temp_mean_trailing,feature_conductivity_max_trailing,feature_conductivity_min_trailing,feature_conductivity_mean_trailing
0,4294500,2014-09-30,18.7,17.2,17.9,NaN,NaN,NaN,44.47616,-73.221517,NaN,NaN,NaN,NaN,NaN,NaN
1,4294500,2014-10-01,17.2,15.8,16.5,179.0,170.0,173.0,44.47616,-73.221517,18.700000,17.200000,17.900000,NaN,NaN,NaN
2,4294500,2014-10-02,17.0,15.8,16.4,177.0,170.0,174.0,44.47616,-73.221517,17.950000,16.500000,17.200000,179.000000,170.000000,173.000000
3,4294500,2014-10-03,17.3,16.5,16.9,179.0,173.0,175.0,44.47616,-73.221517,17.633333,16.266667,16.933333,178.000000,170.000000,173.500000
4,4294500,2014-10-04,17.1,16.4,16.8,191.0,161.0,176.0,44.47616,-73.221517,17.550000,16.325000,16.925000,178.333333,171.000000,174.000000
5,4294500,2014-10-05,16.8,16.2,16.4,184.0,173.0,178.0,44.47616,-73.221517,17.460000,16.340000,16.900000,181.500000,168.500000,174.500000
6,4294500,2014-10-06,16.2,15.6,15.8,184.0,173.0,179.0,44.47616,-73.221517,17.350000,16.316667,16.816667,182.000000,169.400000,175.200000
7,4294500,2014-10-07,16.0,15.4,15.7,186.0,174.0,178.0,44.47616,-73.221517,17.185714,16.214286,16.671429,182.333333,170.000000,175.833333
8,4294500,2014-10-08,16.0,15.2,15.8,193.0,175.0,179.0,44.47616,-73.221517,16.800000,15.957143,16.357143,182.857143,170.571429,176.142857
9,4294500,2014-10-09,15.2,14.6,14.8,186.0,170.0,175.0,44.47616,-73.221517,16.628571,15.871429,16.257143,184.857143,171.285714,177.000000


### Reorder & rename columns for output

In [152]:
# Reorder and rename columns

# Set first columns (explicitly)
first_cols = ["site", "report_date", "latitude", "longitude"]

# Get the rest of the columns in the dataframe, in order
all_cols = usgs_trailing_df.columns.tolist()

# Create new column order with the firs columns + the rest of the columns in the same order
col_order = first_cols + [c for c in all_cols if c not in first_cols]

# Reorder the dataframe
usgs_ordered_df = usgs_trailing_df[col_order]

usgs_ordered_df.columns

Index(['site', 'report_date', 'latitude', 'longitude',
       'feature_water_temp_max', 'feature_water_temp_min',
       'feature_water_temp_mean', 'feature_conductivity_max',
       'feature_conductivity_min', 'feature_conductivity_mean',
       'feature_water_temp_max_trailing', 'feature_water_temp_min_trailing',
       'feature_water_temp_mean_trailing', 'feature_conductivity_max_trailing',
       'feature_conductivity_min_trailing',
       'feature_conductivity_mean_trailing'],
      dtype='object')

### Calculate density of features

In [153]:
# Coverage for USGS features

# Get year from the date
usgs_ordered_df["report_date"] = pd.to_datetime(usgs_ordered_df["report_date"])
usgs_ordered_df["year"] = usgs_ordered_df["report_date"].dt.year

# Columns that must all be non-NaN
# required_cols = ["feature_water_temp"] + cyano_features + ["target_bloom"]
usgs_features = ["feature_water_temp_mean_trailing", "feature_conductivity_mean_trailing"]

for feature in usgs_features:

    yearly_percents = []
    total_valid_days = 0
    total_possible_days = 0

    for year, df_year in usgs_ordered_df.groupby("year"):

        min_date = df_year["report_date"].min()
        max_date = df_year["report_date"].max()

        full_range = pd.date_range(start=min_date, end=max_date, freq="D")

        # Days where this feature is non-NaN
        valid_days = (
            df_year
            .dropna(subset=[feature])
            ["report_date"]
            .dt.normalize()
            .unique()
        )

        percent_valid = len(valid_days) / len(full_range) * 100
        yearly_percents.append(percent_valid)

        total_valid_days += len(valid_days)
        total_possible_days += len(full_range)

    # Straight average across years
    avg_percent = np.mean(yearly_percents)

    # Weighted overall percent
    weighted_percent = (total_valid_days / total_possible_days) * 100

    print(f"\n{feature}")
    # print(f"Average yearly percent: {avg_percent:.2f}%")
    print(f"Weighted overall percent: {weighted_percent:.2f}%")



feature_water_temp_mean_trailing
Weighted overall percent: 99.64%

feature_conductivity_mean_trailing
Weighted overall percent: 99.60%


### Output file

In [154]:
output_df = usgs_ordered_df.copy()
output_df = output_df[output_df.columns.drop(["year"])]

output_df.head()

,site,report_date,latitude,longitude,feature_water_temp_max,feature_water_temp_min,feature_water_temp_mean,feature_conductivity_max,feature_conductivity_min,feature_conductivity_mean,feature_water_temp_max_trailing,feature_water_temp_min_trailing,feature_water_temp_mean_trailing,feature_conductivity_max_trailing,feature_conductivity_min_trailing,feature_conductivity_mean_trailing
0,4294500,2014-09-30,44.47616,-73.221517,18.7,17.2,17.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4294500,2014-10-01,44.47616,-73.221517,17.2,15.8,16.5,179.0,170.0,173.0,18.700000,17.200000,17.900000,NaN,NaN,NaN
2,4294500,2014-10-02,44.47616,-73.221517,17.0,15.8,16.4,177.0,170.0,174.0,17.950000,16.500000,17.200000,179.000000,170.0,173.0
3,4294500,2014-10-03,44.47616,-73.221517,17.3,16.5,16.9,179.0,173.0,175.0,17.633333,16.266667,16.933333,178.000000,170.0,173.5
4,4294500,2014-10-04,44.47616,-73.221517,17.1,16.4,16.8,191.0,161.0,176.0,17.550000,16.325000,16.925000,178.333333,171.0,174.0


### Merge into one unified csv

In [155]:
# Output the unified csv file

output_df.to_csv(output_path, index=False)